# MIE312 Lab 4 - Impulse Turbines

Developed by Prof. Sullivan (pierre.sullivan@utoronto.ca). Suggestions for improvements are welcome.

This notebook follows the Fall 2026 Lab 4 manual. It keeps the **brake-drum radius** and **effective blade radius** separate, uses the measured pressure and flow rate at each operating point, and calculates torque, jet velocity, water power, brake power, experimental efficiency, theoretical efficiency, and the peripheral velocity factor $\phi$.

## Notes
This is the **clean** version.

Use it when you want a blank notebook for your own data while keeping all formulas,
plots, and analysis sections in place.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import pi

plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.precision", 4)

In [ ]:
# Fixed parameters from the lab manual
rho = 997.0          # kg/m^3 (water; adjust if a different lab temperature is specified)
Cv = 0.95
r_drum = 0.025       # m, brake-drum radius used only for torque
r_blade = 0.046      # m, effective turbine radius used for blade speed and phi


def kPa_to_Pa(P_kPa):
    return 1000.0 * P_kPa


def L_min_to_m3_s(Q_L_min):
    return Q_L_min / 1000.0 / 60.0

print(f'Brake-drum radius R_d = {r_drum:.3f} m')
print(f'Effective blade radius r_b = {r_blade:.3f} m')

## Input your measurements
Enter the **net brake-force difference** $F=B-A$, turbine speed, gauge pressure, and flow rate for each operating point. The unloaded row should have $F=0$ but a nonzero (typically maximum) rotational speed.

In [ ]:
# Enter your data here. Pressure and flow rate should be recorded point-by-point.
columns = ['trial', 'F_N', 'omega_rpm', 'P_gauge_kPa', 'Q_L_min']

beta90 = pd.DataFrame({c: [] for c in columns})
beta160 = pd.DataFrame({c: [] for c in columns})

# Tachometer vs stroboscope comparison data
rpm_compare = pd.DataFrame({
    'trial': [],
    'F_N': [],
    'tachometer_rpm': [],
    'stroboscope_rpm': [],
})

beta90, beta160, rpm_compare

## Compute torque, jet speed, blade speed, power, $\phi$, and efficiencies
For each operating point,

$$T=R_dF,\qquad V_b=r_b\omega,$$

where $R_d=0.025$ m is the brake-drum radius and $r_b=0.046$ m is the effective blade radius. Do not interchange these radii.

In [ ]:
def turbine_results(df, beta_deg):
    out = df.copy()
    if len(out) == 0:
        # Add the result columns so later plotting/table cells remain well defined.
        for col in ['P_gauge_Pa', 'Q_m3_s', 'Vj_m_s', 'omega_rad_s', 'Torque_Nm',
                    'Vb_m_s', 'phi', 'BHP_W', 'water_power_W', 'eta_exp', 'eta_th']:
            out[col] = pd.Series(dtype=float)
        return out

    out['P_gauge_Pa'] = out['P_gauge_kPa'].apply(kPa_to_Pa)
    out['Q_m3_s'] = out['Q_L_min'].apply(L_min_to_m3_s)
    out['Vj_m_s'] = np.where(
        out['P_gauge_Pa'] >= 0,
        Cv * np.sqrt(2 * out['P_gauge_Pa'] / rho),
        np.nan,
    )
    out['omega_rad_s'] = out['omega_rpm'] * 2 * np.pi / 60
    out['Torque_Nm'] = r_drum * out['F_N']
    out['Vb_m_s'] = out['omega_rad_s'] * r_blade
    out['phi'] = np.where(out['Vj_m_s'] > 0, out['Vb_m_s'] / out['Vj_m_s'], np.nan)
    out['BHP_W'] = out['omega_rad_s'] * out['Torque_Nm']
    out['water_power_W'] = 0.5 * rho * out['Vj_m_s']**2 * out['Q_m3_s']
    out['eta_exp'] = np.where(out['water_power_W'] > 0, out['BHP_W'] / out['water_power_W'], np.nan)
    out['eta_th'] = 2 * out['phi'] * (1 - out['phi']) * (1 - np.cos(np.deg2rad(beta_deg)))
    return out

res90 = turbine_results(beta90, 90)
res160 = turbine_results(beta160, 160)

print('β = 90°')
display(res90)
print('\nβ = 160°')
display(res160)

## Plot theoretical and experimental efficiencies

In [ ]:
phi_grid = np.linspace(0, 1, 400)
eta90_th_grid = 2 * phi_grid * (1 - phi_grid) * (1 - np.cos(np.deg2rad(90)))
eta160_th_grid = 2 * phi_grid * (1 - phi_grid) * (1 - np.cos(np.deg2rad(160)))

plt.plot(phi_grid, eta90_th_grid, label="Theoretical β=90°")
plt.plot(phi_grid, eta160_th_grid, label="Theoretical β=160°")
if len(res90):
    plt.scatter(res90["phi"], res90["eta_exp"], label="Experimental β=90°")
if len(res160):
    plt.scatter(res160["phi"], res160["eta_exp"], label="Experimental β=160°")
plt.xlabel("Peripheral velocity factor φ")
plt.ylabel("Efficiency η")
plt.title("Theoretical and Experimental Turbine Efficiency")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Experimental-only plot
if len(res90) or len(res160):
    plt.plot(res90["phi"], res90["eta_exp"], marker="o", label="Experimental β=90°")
    plt.plot(res160["phi"], res160["eta_exp"], marker="o", label="Experimental β=160°")
    plt.xlabel("Peripheral velocity factor φ")
    plt.ylabel("Experimental efficiency η")
    plt.title("Experimental Efficiency vs φ")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## Summary / analysis

In [ ]:
def summarize(res, beta):
    if len(res) == 0:
        return {"beta": beta, "phi_at_max_exp": np.nan, "max_exp_eta": np.nan}
    idx = res["eta_exp"].idxmax()
    return {
        "beta": beta,
        "phi_at_max_exp": float(res.loc[idx, "phi"]),
        "max_exp_eta": float(res.loc[idx, "eta_exp"]),
        "rpm_at_max_exp": float(res.loc[idx, "omega_rpm"]),
        "force_at_max_exp": float(res.loc[idx, "F_N"]),
    }

summary = pd.DataFrame([summarize(res90, 90), summarize(res160, 160)])
display(summary)

# Theoretical max occurs at phi = 0.5 for both blade angles
print("Theoretical maximum efficiency occurs at φ = 0.5 for both turbines.")
print(f"Theoretical η_max for β=90°  = {2*0.5*(1-0.5)*(1-np.cos(np.deg2rad(90))):.3f}")
print(f"Theoretical η_max for β=160° = {2*0.5*(1-0.5)*(1-np.cos(np.deg2rad(160))):.3f}")

if len(rpm_compare):
    rpm_compare = rpm_compare.copy()
    rpm_compare["difference_rpm"] = rpm_compare["stroboscope_rpm"] - rpm_compare["tachometer_rpm"]
    rpm_compare["percent_diff_%"] = np.where(
        rpm_compare["tachometer_rpm"] != 0,
        100 * rpm_compare["difference_rpm"] / rpm_compare["tachometer_rpm"],
        np.nan,
    )
    print("\nTachometer vs stroboscope:")
    display(rpm_compare)
    print(f"Mean absolute difference = {rpm_compare['difference_rpm'].abs().mean():.2f} rpm")

In [ ]:
# Optional text summary generator
valid_summary = summary.dropna(subset=['max_exp_eta', 'phi_at_max_exp'])
for _, row in valid_summary.iterrows():
    print(
        f"β={int(row['beta'])}°: max experimental efficiency = "
        f"{row['max_exp_eta']:.3f} at φ = {row['phi_at_max_exp']:.3f}"
    )

if len(valid_summary) == 0:
    print('Enter turbine measurements before generating the experimental-efficiency summary.')